In [19]:
# with open("/Volumes/LaCie/AllExit.txt") as input_file:
#     head = [next(input_file) for _ in range(5000000)]
# #print(head)

In [20]:
# with open('/Users/beto/Documents/Projects/medLLM/d6_test.txt') as input_file:
#     head = [next(input_file) for _ in range(100)]

In [18]:
# # open file
# with open('/Users/beto/Documents/Projects/medLLM/d6_100.txt', 'w+') as f:
     
#     # write elements of list
#     for items in head:
#         f.write('%s' %items)
     
#     print("File written successfully")
 
 
# # close the file
# f.close()

In [15]:
#### FOR INSTALLATION
# pip3 install -U ray[default]

In [25]:
def get_words(a_line):
    words = a_line.split(' : ')
    return words
    


In [26]:
fst = get_words("000 : 4526366 : 5 : 156249974 : 3 : 89251952 : 4 : 89251952")
fst

['000', '4526366', '5', '156249974', '3', '89251952', '4', '89251952']

In [30]:
import numpy as np
import pandas as pd

In [39]:
x = np.loadtxt('d6_test.txt',delimiter = ':', dtype=str)
df = pd.DataFrame(x)
df.columns = ["one", "two", "three", "four", "five", "six", "seven", "eight"]
df

,one,two,three,four,five,six,seven,eight
0,-1,156250000,7,156249999,2,0,2,0
1,-1,156250000,7,156249999,2,0,2,0
2,-1,156250000,7,156249999,2,0,2,0
3,-1,156250000,7,156249999,2,0,2,0
4,-1,142324024,7,89251951,2,0,2,0
...,...,...,...,...,...,...,...,...
4999995,101001011,1,22,152566630,6,152566630,10,152566630
4999996,1010010110,1,31,88540271,6,88540271,11,88540271
4999997,10100101100,3,27,8938152,6,8938152,11,8938152
4999998,1010011,228,11,148861235,5,27187721,8,88505289


In [40]:
(df.groupby("one").count()).sort_values(by="two", ascending=False)["two"][0:20]

one
-1          2189
00111       1729
00010       1729
00110       1729
0001000     1729
01100       1729
011000      1729
100110      1729
10011       1729
001110      1729
10000       1729
100000      1729
01101       1729
10100       1729
011010      1729
10010       1729
01110       1729
0010000     1729
001000      1729
00100       1729
Name: two, dtype: int64

In [2]:
import ray
#ray.shutdown()
ray.init()

2024-04-09 17:24:13,913	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.11
Ray version:,2.10.0
Dashboard:,http://127.0.0.1:8265


In [42]:
from typing import Any, Dict
import ray

def to_lower(row: Dict[str, Any]) -> Dict[str, Any]:
    row["text"] = row["text"].lower()
    return row

def get_first_n(row: Dict[str, Any]):
    return row

ds = (
    ray.data.read_text("/Users/beto/Documents/Projects/medLLM/d6_test.txt")
    .map(get_first_n)
)

ds.show(100)

2024-04-09 18:18:50,058	INFO streaming_executor.py:115 -- Starting execution of Dataset. Full log is in /tmp/ray/session_2024-04-09_17-24-12_355964_10206/logs/ray-data.log
2024-04-09 18:18:50,059	INFO streaming_executor.py:116 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadText] -> TaskPoolMapOperator[Map(get_first_n)] -> LimitOperator[limit=100]



- ReadText->SplitBlocks(100) 1:   0%|          | 0/1 [00:00<?, ?it/s]

- Map(get_first_n) 2:   0%|          | 0/1 [00:00<?, ?it/s]

- limit=100 3:   0%|          | 0/1 [00:00<?, ?it/s]

Running 0:   0%|          | 0/1 [00:00<?, ?it/s]

{'text': '1010001000111 : 4 : 63 : 152665778 : 6 : 152665722 : 12 : 152665722'}
{'text': '1010001000111001 : 1 : 65 : 152507378 : 6 : 152507378 : 12 : 152507378'}
{'text': '101000100011101 : 1 : 47 : 152499266 : 6 : 152499266 : 12 : 152499266'}
{'text': '10100010001111 : 2 : 46 : 152503998 : 6 : 152503998 : 12 : 152503998'}
{'text': '1010001001 : 277 : 28 : 152729782 : 5 : 50533458 : 10 : 50533458'}
{'text': '10100010010 : 1 : 36 : 49161231 : 6 : 49161231 : 12 : 49161231'}
{'text': '1010001001001 : 3 : 48 : 153377910 : 6 : 148975122 : 12 : 148975122'}
{'text': '10100010011 : 7 : 45 : 155339348 : 6 : 137466949 : 12 : 137466949'}
{'text': '101000100110 : 3 : 49 : 49318102 : 6 : 49212646 : 12 : 49212646'}
{'text': '1010001001100 : 2 : 48 : 36525100 : 6 : 36525100 : 11 : 36525100'}
{'text': '101000100110110 : 1 : 44 : 49769761 : 6 : 49769761 : 12 : 49769761'}
{'text': '10100010011101 : 2 : 46 : 152727754 : 6 : 152727754 : 12 : 152727754'}
{'text': '1010001001111 : 3 : 39 : 109158987 : 6 : 

In [93]:
# Define a function to process a chunk of data
@ray.remote
def process_chunk(chunk):
    # Process each line in the chunk
    counts = {}
    for line in chunk:
        line = line[0]
        line = str(line)
        line = line.split(":")
        line = line[0]
        pattern = line.strip()  # Extract the pattern from the first column
        counts[pattern] = counts.get(pattern, 0) + 1  # Count occurrences of each pattern
    return counts

# Define a function to merge counts from different chunks
@ray.remote
def merge_counts(*counts):
    merged_counts = {}
    for count in counts:
        for pattern, freq in count.items():
            merged_counts[pattern] = merged_counts.get(pattern, 0) + freq
    return merged_counts

In [101]:


# Load the CSV file in chunks
chunk_size = 100000 # Adjust as needed
chunks = pd.read_csv('/Volumes/LaCie/AllExit.txt', chunksize=chunk_size)

# Process each chunk in parallel
count_futures = [process_chunk.remote(chunk.values.tolist()) for chunk in chunks]
# count_futures = [process_chunk.remote(chunk) for chunk in chunks]
# print("IM HERE ********")
# print(count_futures)

# Merge the counts from all chunks
merged_counts = ray.get(merge_counts.remote(*count_futures))
print(merge_counts)

# Find the top 20 most frequent patterns
top_20_patterns = sorted(merged_counts.items(), key=lambda x: x[1], reverse=True)[:20]

# Print the top 20 patterns
for pattern, frequency in top_20_patterns:
    print(f"Pattern: {pattern}, Frequency: {frequency}")



(raylet) Spilled 3271 MiB, 1053 objects, write throughput 1502 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.
(raylet) Spilled 4907 MiB, 1575 objects, write throughput 1631 MiB/s.
(raylet) Spilled 9816 MiB, 3137 objects, write throughput 1833 MiB/s.
(raylet) Spilled 17996 MiB, 5743 objects, write throughput 2009 MiB/s.
(raylet) Spilled 34337 MiB, 10957 objects, write throughput 2402 MiB/s.
(raylet) Spilled 67061 MiB, 21159 objects, write throughput 2812 MiB/s.
(raylet) Spilled 132473 MiB, 41709 objects, write throughput 2432 MiB/s.
(raylet) Spilled 263268 MiB, 82571 objects, write throughput 2217 MiB/s.


Pattern: -1, Frequency: 3565215
Pattern: 0100, Frequency: 2585386
Pattern: 00000, Frequency: 2585177
Pattern: 00100, Frequency: 2584979
Pattern: 01000, Frequency: 2584891
Pattern: 10000, Frequency: 2584879
Pattern: 00111, Frequency: 2584842
Pattern: 00010, Frequency: 2584823
Pattern: 01100, Frequency: 2584713
Pattern: 00110, Frequency: 2584702
Pattern: 10111, Frequency: 2584701
Pattern: 00011, Frequency: 2584655
Pattern: 0111, Frequency: 2584654
Pattern: 01111, Frequency: 2584578
Pattern: 11000, Frequency: 2584537
Pattern: 00101, Frequency: 2584513
Pattern: 01011, Frequency: 2584508
Pattern: 11011, Frequency: 2584440
Pattern: 00001, Frequency: 2584438
Pattern: 10100, Frequency: 2584437


In [102]:
# # Shut down Ray
ray.shutdown()

In [105]:
type(merged_counts)

dict

In [106]:
try: 
    geeky_file = open('counting_d6.txt', 'wt') 
    geeky_file.write(str(merged_counts)) 
    geeky_file.close() 
  
except: 
    print("Unable to write to file")

In [109]:
import pickle

def pet_save(file_name, dictionary):
    with open(file_name + '.pickle', 'wb') as f:
        pickle.dump(dictionary, f, pickle.HIGHEST_PROTOCOL)

def digimon_load(pet_name):
    with open(pet_name + '.pickle', 'rb') as f:
        return pickle.load(f)

In [111]:
pet_save("counting_d6", merged_counts)

In [112]:
new_dict = digimon_load("counting_d6")
new_dict

{'-1': 3565215,
 '000': 2537470,
 '0000': 2583786,
 '00000': 2585177,
 '000000': 2584095,
 '0000000': 2578804,
 '00000000': 2520638,
 '00000001': 2235086,
 '0000001': 2486355,
 '00000010': 2354525,
 '00000011': 2160002,
 '000001': 2579398,
 '0000010': 2558805,
 '00000100': 2338225,
 '000001000': 1878024,
 '00000101': 2125436,
 '000001011': 1434538,
 '0000011': 2515749,
 '00000110': 2249318,
 '00000111': 2167150,
 '00001': 2584438,
 '000010': 2582743,
 '0000100': 2548594,
 '00001000': 2308173,
 '000010000': 1795010,
 '0000100011': 683381,
 '00001001': 2157605,
 '00001001010': 482349,
 '0000101': 2471828,
 '00001010': 2298335,
 '0000101010': 1284174,
 '00001010101': 683316,
 '00001011': 2198268,
 '000010110': 1579657,
 '000011': 2578743,
 '0000110': 2525346,
 '00001100': 2285711,
 '000011000': 1529128,
 '00001101': 2177052,
 '0000111': 2508855,
 '00001110': 2158301,
 '000011100': 1471288,
 '0000111100': 665242,
 '0001': 2583446,
 '00010': 2584823,
 '000100': 2583205,
 '0001000': 2529125,